# Front-Running Detection: Visualization & Real-Time Testing

This notebook visualizes the predictive MEV detection model and tests it in real-time simulation.

In [1]:
# Imports for the notebook - run this first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc, 
                             precision_recall_curve, roc_auc_score)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Notebook utilities
from IPython.display import clear_output, display
import time

# For live monitoring
import asyncio
from web3 import Web3

# Style settings
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10


In [2]:
# the method:

# passively run collect_data.py to continiously get new data. don't turn it off. just let it cook if yk what i mean.
# run flashboys_analysis.py to generate a csv containing the analysis from the journal paper. this is the basis for our model.
# then, run this ipynb to train the model and assess it's predictive performance. 
# db_inspect.py is for checking the stats of the db and also to see if it has fully sequantial data. if not, then it call functions from collect_data.py
# to fill in the gaps. 

# why train a model if we have the hard coded algo? this model focuses on prediction -- before the front running actually happens.
# it's performance is checked to see whether or not they agree (the hard coded algo and the model)
# that's where our work stands out

# what type of front running does it predict? this is for detecting gas auctions
# tf are gas auctions?? i'll tell ya: it's when multiple bots are competing by bidding higher gas prices to get their transaction mined first in the same block
# why tf would they do that: aight chill tf out, i'll tell ya:

# let's say there was a bot called... let's say "67," and it spots an arbitrage opportunity. it submits tx with 50 gwei gas. bot... let's call it "41," sees
# bot 67's tx in the mempool. bot 41 then submits a competing tx with 60 gwer gas. then there's some other bot that came out of basically nowhere that 
# then bids 70 gwei gas. 

# smooch 💋

In [ ]:
# Load RAW transaction data + FlashBoys labels
import sqlite3

print(f"📊 Loading data...")
print(f"{'='*70}")

# 1. Load RAW transactions from database (what model sees in real-time)
print("Loading RAW transactions from crypto_data.db...")
conn = sqlite3.connect('data/crypto_data.db')
query = """
SELECT 
    block_number,
    transaction_hash,
    transaction_index,
    from_address,
    to_address,
    value,
    gas_price,
    gas_limit,
    gas_used,
    timestamp,
    nonce
FROM transactions
ORDER BY block_number, transaction_index
"""
df_raw = pd.read_sql_query(query, conn)
conn.close()
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])
print(f"  ✅ Loaded {len(df_raw):,} raw transactions")

# 2. Load MEV labels (gas auction detection: same sender+nonce replacements)
print("Loading MEV labels (gas auction detection)...")
df_labels = pd.read_csv('data/mev_labels.csv')
print(f"  ✅ Loaded {len(df_labels):,} labeled transactions")

# 3. Match them up by transaction hash
print("\nMatching raw data with labels...")
df = df_raw.merge(
    df_labels[['transaction_hash', 'is_mev']], 
    on='transaction_hash',
    how='left'  # Keep ALL transactions from database
)

# Fill NaN values: transactions without labels are non-MEV
df['is_mev'] = df['is_mev'].fillna(0).astype(int)

print(f"  ✅ Total transactions: {len(df):,}")
print(f"  ✅ MEV transactions: {df['is_mev'].sum():,}")
print(f"  MEV rate: {df['is_mev'].mean()*100:.4f}%")

# Add basic computed features (safe - no future leakage)
df['gas_price_gwei'] = df['gas_price'] / 1e9
df['value_eth'] = df['value'] / 1e18

print(f"\n{'='*70}")
print(f"📊 DATASET READY")
print(f"{'='*70}")
print(f"Features: RAW transaction data (gas prices, values, etc.)")
print(f"Labels: Gas auction detection (same sender+nonce replacements)")
print(f"Goal: Predict MEV using ONLY historical features")
print(f"{'='*70}\n")

# Display sample
print("Sample (RAW features + MEV labels):")
df[['block_number', 'transaction_hash', 'gas_price', 'value', 'is_mev']].head(10)

📊 Loading data...
Loading RAW transactions from crypto_data.db...
  ✅ Loaded 833,069 raw transactions
Loading FlashBoys labels (answer key) from flashboys_analysis.csv...
  ✅ Loaded 833,069 raw transactions
Loading FlashBoys labels (answer key) from flashboys_analysis.csv...
  ✅ Loaded 833,068 labeled transactions

Matching raw data with labels...
  Strategy: LEFT JOIN - all raw transactions, label MEV where detected
  ✅ Loaded 833,068 labeled transactions

Matching raw data with labels...
  Strategy: LEFT JOIN - all raw transactions, label MEV where detected
  ✅ Total transactions: 833,069
  ✅ Labeled by FlashBoys: 833,068
  ✅ MEV transactions: 254,799
  MEV rate: 30.59%

📊 DATASET READY
Features: RAW transaction data (gas prices, values, etc.)
Labels: FlashBoys post-hoc analysis (is_mev_auction)
Goal: Predict FlashBoys labels using ONLY historical features

Sample (RAW features + FlashBoys labels):
  ✅ Total transactions: 833,069
  ✅ Labeled by FlashBoys: 833,068
  ✅ MEV transactions

,block_number,transaction_hash,gas_price,value,is_mev_auction
0,23752420,402c10e96486ddd6f816e9917c1cfde5ecb0d83261797d...,12841652931,2.375242e-11,1
1,23752420,d226f44de4708275ba0aabc9710816b50b0f95ed9084d7...,3395238656,0.000000e+00,1
2,23752420,9fd98faaccd9c787c0fb0c5c98169181d75cf459941d54...,336419131,0.000000e+00,1
3,23752420,e18c7c32d88329307067cffef76e0ff8cae0d64bb8b20c...,1646832813,1.184400e+01,1
4,23752420,440050bc50b9de6bc891831656b7ec443b263595fc0d04...,1646832813,1.570000e-02,1
5,23752420,1a456490dd74db356a17fbaf8222bd78ae0687e5de3eae...,3000000000,0.000000e+00,1
6,23752420,aa90f64c6efaea56e2a370bde9872d4b345843aedf6c54...,638210886,1.000000e-03,1
7,23752420,3d0eafdf86ebe785a70dc4fd0270c009f8a19466b658a3...,1139011844,0.000000e+00,1
8,23752420,e5d65beaf0e21cb75293b39a2ddbe8365e6eb316d7b2b2...,1646832813,0.000000e+00,1
9,23752420,70d657ccdc5a685f3bab8e4a3acac2fff687ed0e3e1c58...,146832813,1.762579e-09,1


In [ ]:
print("Engineering PREDICTIVE features (NO DATA LEAKAGE)...")
print("Building SINGLE model that predicts MEV at every transaction\n")

# SPEED OPTIMIZATION: Set to True for faster prototyping (uses subset of data)
FAST_MODE = True  # Change to True for 5-10x faster training

if FAST_MODE:
    print("⚡ FAST MODE ENABLED - Using 20% of data for quick iteration")
    print("   (Set FAST_MODE = False for full training)\n")

df_sorted = df.sort_values(['block_number', 'transaction_index']).reset_index(drop=True)

# Sample data if in fast mode
if FAST_MODE:
    sample_size = int(len(df_sorted) * 0.2)
    df_sorted = df_sorted.sample(n=sample_size, random_state=42).sort_values(['block_number', 'transaction_index']).reset_index(drop=True)
    print(f"Using {len(df_sorted):,} transactions (20% sample)\n")

print(f"{'='*70}")
print(f"Building SINGLE predictive model")
print(f"{'='*70}\n")

features_per_tx = []
labels = []
valid_indices = []

for idx, row in df_sorted.iterrows():
    # ONLY USE PAST DATA - NO FUTURE INFORMATION
    features = {
        'gas_price': row['gas_price'],
        'gas_limit': row['gas_limit'],
        'value': row['value'],
        'gas_price_gwei': row['gas_price_gwei'],
        'gas_price_to_limit_ratio': row['gas_price'] / (row['gas_limit'] + 1),
        'value_to_gas_ratio': row['value'] / (row['gas_price'] + 1),
    }
    
    # Historical features (looking backward - safe!)
    # Use smaller window in fast mode for speed
    lookback_window = 50 if FAST_MODE else 100
    
    if idx >= lookback_window:
        recent = df_sorted.iloc[idx-lookback_window:idx]  # Last N txs BEFORE this one
        
        # Gas price patterns
        features['recent_avg_gas'] = recent['gas_price'].mean()
        features['recent_max_gas'] = recent['gas_price'].max()
        features['recent_min_gas'] = recent['gas_price'].min()
        features['recent_gas_std'] = recent['gas_price'].std()
        
        # Gas price volatility (sign of competition)
        features['gas_volatility'] = features['recent_gas_std'] / (features['recent_avg_gas'] + 1)
        features['gas_vs_recent_avg'] = row['gas_price'] / (features['recent_avg_gas'] + 1)
        features['gas_vs_recent_max'] = row['gas_price'] / (features['recent_max_gas'] + 1)
        
        # Transaction density (congestion indicator)
        features['recent_tx_density'] = len(recent) / lookback_window  # Should be ~1.0 normally
        
        # High gas concentration (multiple high-gas txs = brewing auction)
        high_gas_threshold = features['recent_avg_gas'] * 1.5
        features['recent_high_gas_count'] = (recent['gas_price'] > high_gas_threshold).sum()
        features['recent_high_gas_ratio'] = features['recent_high_gas_count'] / lookback_window
        
        # Consecutive high-gas transactions (escalation pattern)
        recent_gas_prices = recent['gas_price'].values
        consecutive_high = 0
        max_consecutive = 0
        for gas in recent_gas_prices[-20:]:  # Last 20 txs
            if gas > high_gas_threshold:
                consecutive_high += 1
                max_consecutive = max(max_consecutive, consecutive_high)
            else:
                consecutive_high = 0
        features['max_consecutive_high_gas'] = max_consecutive
        
        # Gas price momentum (acceleration indicator)
        if len(recent) >= 20:
            recent_10 = recent.iloc[-10:]['gas_price'].mean()
            previous_10 = recent.iloc[-20:-10]['gas_price'].mean()
            features['gas_momentum'] = (recent_10 - previous_10) / (previous_10 + 1)
        else:
            features['gas_momentum'] = 0
            
    else:
        # Not enough history - use defaults
        features.update({
            'recent_avg_gas': row['gas_price'],
            'recent_max_gas': row['gas_price'],
            'recent_min_gas': row['gas_price'],
            'recent_gas_std': 0,
            'gas_volatility': 0,
            'gas_vs_recent_avg': 1,
            'gas_vs_recent_max': 1,
            'recent_tx_density': 1,
            'recent_high_gas_count': 0,
            'recent_high_gas_ratio': 0,
            'max_consecutive_high_gas': 0,
            'gas_momentum': 0
        })
    
    # LABEL: Use gas auction detection (transaction replacement)
    # MEV = same sender replaced tx with higher gas (competitive bidding)
    # Model will learn to predict this using ONLY historical features
    label = int(row['is_mev'])
    
    labels.append(label)
    features_per_tx.append(features)
    valid_indices.append(idx)

X = pd.DataFrame(features_per_tx)
y = pd.Series(labels)

print(f"Total samples: {len(X):,}")
print(f"MEV rate: {y.mean()*100:.1f}% ({y.sum()} MEV transactions found)")
print(f"Features: {len(X.columns)}")

if y.sum() == 0:
    print("\n⚠️  NO MEV DETECTED!")
    print("   Possible reasons:")
    print("   1. Sample size too small (only 20% of data)")
    print("   2. Gas auction detection logic too strict")
    print("   3. No actual gas auctions in this time period")
    print("\n   Try: Set FAST_MODE = False to use full dataset\n")

# Time-based split (70% train, 30% test)
split_idx = int(len(X) * 0.7)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Verify temporal ordering
assert split_idx < len(valid_indices), "Split index out of range"
train_max_idx = valid_indices[split_idx - 1]
test_min_idx = valid_indices[split_idx]
assert train_max_idx < test_min_idx, "TEMPORAL LEAKAGE: Test data before train data!"
print(f"✅ Temporal ordering verified (train ends at idx {train_max_idx}, test starts at {test_min_idx})")

print(f"\nTraining on {len(X_train):,} txs, testing on {len(X_test):,} txs")

# Train model (optimized for M4 Pro - parallel processing)
print(f"Training model with parallel processing (using all CPU cores)...\n")

from sklearn.ensemble import HistGradientBoostingClassifier

# HistGradientBoostingClassifier is MUCH faster than GradientBoostingClassifier
# - Uses histogram-based algorithm (10-100x faster on large datasets)
# - Native support for missing values
# - Parallel training across all cores
model = HistGradientBoostingClassifier(
    max_iter=200,           # equivalent to n_estimators
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    max_bins=255,           # trade-off: lower = faster, higher = more accurate
    early_stopping=True,    # stops when validation score stops improving
    n_iter_no_change=10,    # patience for early stopping
    validation_fraction=0.1,
    verbose=0
)

# Handle class imbalance
if y_train.sum() == 0:
    print("⚠️  WARNING: No MEV transactions found in training data!")
    print("   This likely means the gas auction detection logic needs adjustment.")
    print("   Training anyway with all zeros (model will predict 'no MEV' for everything)...\n")
    sample_weights = None
else:
    class_weight = len(y_train) / (2 * np.bincount(y_train))
    sample_weights = np.where(y_train == 1, class_weight[1], class_weight[0])

import time
start_time = time.time()
if sample_weights is not None:
    model.fit(X_train, y_train, sample_weight=sample_weights)
else:
    model.fit(X_train, y_train)
train_time = time.time() - start_time

print(f"✅ Model trained in {train_time:.1f} seconds!")

# Store results
results = {
    'model': model,
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'feature_names': X.columns.tolist(),
    'valid_indices': valid_indices,
    'split_idx': split_idx
}

print(f"\n{'='*70}")
print("✅ MODEL TRAINED - NO DATA LEAKAGE")
print(f"{'='*70}\n")

# Get predictions on test set
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

Engineering PREDICTIVE features (NO DATA LEAKAGE)...
Building SINGLE model that predicts MEV at every transaction

⚡ FAST MODE ENABLED - Using 20% of data for quick iteration
   (Set FAST_MODE = False for full training)

Using 166,613 transactions (20% sample)

Building SINGLE predictive model

Total samples: 166,613
MEV rate: 30.6% (50959 MEV transactions found)
Features: 18
✅ Temporal ordering verified (train ends at idx 116628, test starts at 116629)

Training on 116,629 txs, testing on 49,984 txs
Training model with parallel processing (using all CPU cores)...

Total samples: 166,613
MEV rate: 30.6% (50959 MEV transactions found)
Features: 18
✅ Temporal ordering verified (train ends at idx 116628, test starts at 116629)

Training on 116,629 txs, testing on 49,984 txs
Training model with parallel processing (using all CPU cores)...

✅ Model trained in 1.8 seconds!

✅ MODEL TRAINED - NO DATA LEAKAGE

✅ Model trained in 1.8 seconds!

✅ MODEL TRAINED - NO DATA LEAKAGE



In [ ]:
print("="*70)
print("ANALYZING PREDICTION LEAD TIME")
print("="*70)
print("\nCalculating how far ahead model predicts MEV...\n")

# Confidence threshold for "prediction"
CONFIDENCE_THRESHOLD = 0.70

# Get test data indices
test_valid_indices = valid_indices[split_idx:]

# Analyze lead time for each MEV event
lead_times = []
mev_detected_early = 0
mev_total = 0

for i, (test_idx, actual_label, prob) in enumerate(zip(test_valid_indices, y_test, y_pred_proba)):
    if actual_label == 1:  # This is an MEV transaction
        mev_total += 1
        
        # Look backward: how far back did model cross threshold?
        # Search previous N transactions
        lookback = 100  # Look back up to 100 txs (~2 seconds)
        lead_tx_count = 0
        
        for j in range(1, min(lookback, i + 1)):
            prev_idx = i - j
            prev_prob = y_pred_proba[prev_idx]
            
            if prev_prob >= CONFIDENCE_THRESHOLD:
                # Model was confident J transactions ago!
                lead_tx_count = j
            else:
                # Found where confidence dropped below threshold
                break
        
        if lead_tx_count > 0:
            lead_times.append(lead_tx_count)
            mev_detected_early += 1

# Basic stats
print(f"Total MEV events in test set: {mev_total}")
print(f"MEV events predicted early (>{CONFIDENCE_THRESHOLD:.0%} confidence): {mev_detected_early}")
print(f"Early detection rate: {mev_detected_early/mev_total*100:.1f}%\n")

if len(lead_times) > 0:
    print(f"Lead Time Statistics:")
    print(f"  Mean lead time: {np.mean(lead_times):.1f} transactions (~{np.mean(lead_times)/50:.2f} seconds)")
    print(f"  Median lead time: {np.median(lead_times):.1f} transactions (~{np.median(lead_times)/50:.2f} seconds)")
    print(f"  Max lead time: {np.max(lead_times)} transactions (~{np.max(lead_times)/50:.2f} seconds)")
    print(f"  Min lead time: {np.min(lead_times)} transactions (~{np.min(lead_times)/50:.2f} seconds)")

# Visualize lead time distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Lead time distribution
ax = axes[0, 0]
if len(lead_times) > 0:
    ax.hist(lead_times, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(lead_times), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {np.mean(lead_times):.1f} txs')
    ax.axvline(np.median(lead_times), color='orange', linestyle='--', linewidth=2,
               label=f'Median: {np.median(lead_times):.1f} txs')
ax.set_xlabel('Lead Time (transactions)', fontweight='bold', fontsize=12)
ax.set_ylabel('Frequency', fontweight='bold', fontsize=12)
ax.set_title('Distribution of Prediction Lead Times', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 2: Confidence distribution
ax = axes[0, 1]
ax.hist(y_pred_proba[y_test==1], bins=50, alpha=0.7, color='red', edgecolor='black', label='MEV txs')
ax.hist(y_pred_proba[y_test==0], bins=50, alpha=0.7, color='blue', edgecolor='black', label='Normal txs')
ax.axvline(CONFIDENCE_THRESHOLD, color='black', linestyle='--', linewidth=2, 
           label=f'Threshold: {CONFIDENCE_THRESHOLD:.0%}')
ax.set_xlabel('Model Confidence', fontweight='bold', fontsize=12)
ax.set_ylabel('Frequency', fontweight='bold', fontsize=12)
ax.set_title('Confidence Distribution: MEV vs Normal Transactions', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 3: Performance metrics
ax = axes[1, 0]
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
}
colors = ['#2ecc71' if v > 0.7 else '#f39c12' if v > 0.6 else '#e74c3c' for v in metrics.values()]
bars = ax.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='black', linewidth=2)
ax.set_ylabel('Score', fontweight='bold', fontsize=12)
ax.set_title('Model Performance Metrics', fontweight='bold', fontsize=14)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2%}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# Plot 4: ROC Curve
ax = axes[1, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color='#e74c3c', lw=3, label=f'ROC (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='#95a5a6', lw=2, linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate', fontweight='bold', fontsize=12)
ax.set_ylabel('True Positive Rate', fontweight='bold', fontsize=12)
ax.set_title('ROC Curve', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n{'='*70}")
print(f"🎯 KEY INSIGHT:")
print(f"{'='*70}")
if len(lead_times) > 0:
    print(f"Model provides ~{np.mean(lead_times)/50:.2f} seconds of advance warning on average!")
    print(f"At {CONFIDENCE_THRESHOLD:.0%} confidence threshold, detects {mev_detected_early/mev_total*100:.1f}% of MEV early")
else:
    print(f"No early detections at {CONFIDENCE_THRESHOLD:.0%} threshold - try lowering threshold")
print(f"{'='*70}\n")

In [ ]:
# Feature importance plot
print(f"Analyzing feature importance...\n")

# For HistGradientBoostingClassifier, use permutation importance as a more reliable method
from sklearn.inspection import permutation_importance

print("Computing feature importance (this may take a moment)...")
perm_importance = permutation_importance(model, X_test, y_test, 
                                         n_repeats=10, random_state=42, n_jobs=-1)

feature_importance = pd.DataFrame({
    'feature': results['feature_names'],
    'importance': perm_importance.importances_mean
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if imp > 0.08 else '#f39c12' if imp > 0.05 else '#3498db' 
          for imp in feature_importance['importance']]
bars = ax.barh(feature_importance['feature'], feature_importance['importance'], 
               color=colors, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Importance', fontweight='bold', fontsize=12)
ax.set_title(f'Feature Importance: What Predicts MEV?\n(NO DATA LEAKAGE - Pure Prediction)', 
             fontweight='bold', fontsize=14)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (idx, row) in enumerate(feature_importance.iterrows()):
    ax.text(row['importance'], i, f" {row['importance']:.4f}", 
            va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n{'='*70}")
print(f"TOP 5 PREDICTIVE FEATURES (NO LEAKAGE):")
print(f"{'='*70}")
for i, row in feature_importance.head(5).iterrows():
    print(f"{i+1}. {row['feature']:35s}: {row['importance']:.2%}")
print(f"{'='*70}\n")

# Verify no 'recent_mev_rate' in features
if 'recent_mev_rate' in feature_importance['feature'].values:
    print("⚠️  WARNING: recent_mev_rate found in features - DATA LEAKAGE!")
else:
    print("✅ VERIFIED: No 'recent_mev_rate' feature - Clean predictive model!")
print()

In [ ]:
# Additional ROC and PR curves
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc_detailed = auc(fpr, tpr)

precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall_curve, precision_curve)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#e74c3c', lw=3, 
             label=f'Model (AUC = {roc_auc_detailed:.4f})')
axes[0].plot([0, 1], [0, 1], color='#95a5a6', lw=2, linestyle='--', label='Random Guess')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate', fontweight='bold', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontweight='bold', fontsize=11)
axes[0].set_title(f'ROC Curve', fontweight='bold', fontsize=14)
axes[0].legend(loc='lower right', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall_curve, precision_curve, color='#3498db', lw=3,
             label=f'Model (AUC = {pr_auc:.4f})')
axes[1].axhline(y=y_test.mean(), color='#95a5a6', lw=2, linestyle='--', 
                label=f'Baseline ({y_test.mean():.2%})')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall', fontweight='bold', fontsize=11)
axes[1].set_ylabel('Precision', fontweight='bold', fontsize=11)
axes[1].set_title(f'Precision-Recall Curve', 
                 fontweight='bold', fontsize=14)
axes[1].legend(loc='upper right', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDetailed Curve Analysis:")
print(f"  ROC-AUC: {roc_auc_detailed:.4f} (1.0 = perfect, 0.5 = random)")
print(f"  PR-AUC: {pr_auc:.4f}")
print(f"  Baseline: {y_test.mean():.2%} (random prediction rate)")
print(f"\n✅ Model provides early warning before MEV events!")

## 9. Real-Time Prediction Simulation

**This cell simulates real-time MEV detection:**
- Streams through test data as if transactions are arriving live
- Shows prediction probability and actual label
- **GREEN** = Correct prediction ✅
- **RED** = Incorrect prediction ❌
- Displays running accuracy

In [ ]:
import time
from IPython.display import clear_output

# Real-time simulation
print("🚀 Starting Real-Time MEV Detection Simulation...")
print("   Streaming through test data as if transactions arrive live\n")
time.sleep(2)

# Sample a subset for demonstration (100 transactions)
n_samples = 100
sample_indices = np.random.choice(len(X_test), n_samples, replace=False)
sample_indices = np.sort(sample_indices)  # Keep temporal order

X_sample = X_test.iloc[sample_indices]
y_sample = y_test.iloc[sample_indices]

correct_predictions = 0
total_predictions = 0

predictions_log = []

for i, (idx, features) in enumerate(X_sample.iterrows()):
    # Make prediction
    pred_proba = model.predict_proba(features.values.reshape(1, -1))[0, 1]
    pred_label = int(pred_proba >= 0.5)
    true_label = y_sample.iloc[i]
    
    # Check if correct
    is_correct = (pred_label == true_label)
    if is_correct:
        correct_predictions += 1
    total_predictions += 1
    
    # Store for visualization
    predictions_log.append({
        'pred_proba': pred_proba,
        'pred_label': pred_label,
        'true_label': true_label,
        'correct': is_correct
    })
    
    # Display update every 10 transactions
    if (i + 1) % 10 == 0 or i < 5:
        clear_output(wait=True)
        
        print("="*70)
        print("🔍 REAL-TIME MEV DETECTION MONITOR")
        print("="*70)
        print(f"\nTransaction #{i+1}/{n_samples}")
        print(f"Running Accuracy: {correct_predictions/total_predictions*100:.1f}% ({correct_predictions}/{total_predictions})")
        print(f"\n{'─'*70}")
        
        # Show last 5 predictions
        print(f"\n📊 Last 5 Predictions:")
        print(f"{'─'*70}")
        for j in range(max(0, i-4), i+1):
            pred = predictions_log[j]
            status = "✅ CORRECT" if pred['correct'] else "❌ WRONG"
            color = "🟢" if pred['correct'] else "🔴"
            label_str = "MEV" if pred['true_label'] == 1 else "NORMAL"
            pred_str = "MEV" if pred['pred_label'] == 1 else "NORMAL"
            
            print(f"{color} TX #{j+1:3d} | Predicted: {pred_str:6s} ({pred['pred_proba']*100:5.1f}%) | "
                  f"Actual: {label_str:6s} | {status}")
        
        print(f"{'─'*70}\n")
        time.sleep(0.3)

# Final summary
clear_output(wait=True)
print("="*70)
print("✅ SIMULATION COMPLETE")
print("="*70)
print(f"\nTotal Transactions Processed: {total_predictions}")
print(f"Correct Predictions: {correct_predictions}")
print(f"Incorrect Predictions: {total_predictions - correct_predictions}")
print(f"\n🎯 Real-Time Accuracy: {correct_predictions/total_predictions*100:.2f}%")
print(f"\n{'─'*70}")

# Prediction distribution
pred_df = pd.DataFrame(predictions_log)
mev_detected = pred_df['pred_label'].sum()
mev_actual = pred_df['true_label'].sum()

print(f"\n📈 Detection Statistics:")
print(f"  MEV Transactions Detected: {mev_detected}")
print(f"  Actual MEV Transactions: {mev_actual}")
print(f"  Detection Rate: {pred_df[pred_df['true_label']==1]['correct'].mean()*100:.1f}%")
print(f"  False Positive Rate: {pred_df[pred_df['true_label']==0]['pred_label'].mean()*100:.1f}%")
print("="*70)

# Visualize predictions
fig, ax = plt.subplots(figsize=(14, 6))

correct_preds = pred_df[pred_df['correct']]
wrong_preds = pred_df[~pred_df['correct']]

ax.scatter(correct_preds.index, correct_preds['pred_proba'], 
           c='green', s=50, alpha=0.6, label='Correct Predictions', marker='o')
ax.scatter(wrong_preds.index, wrong_preds['pred_proba'], 
           c='red', s=80, alpha=0.8, label='Wrong Predictions', marker='X')

ax.axhline(y=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
ax.fill_between(range(len(pred_df)), 0.5, 1.0, alpha=0.1, color='red', label='MEV Zone')
ax.fill_between(range(len(pred_df)), 0, 0.5, alpha=0.1, color='blue', label='Normal Zone')

ax.set_xlabel('Transaction Number', fontweight='bold', fontsize=12)
ax.set_ylabel('MEV Probability', fontweight='bold', fontsize=12)
ax.set_title('Real-Time Prediction Results: Correct vs Incorrect', 
             fontweight='bold', fontsize=14)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Model is ready for production MEV detection!")